# Spatial Joins Exercises

Here\'s a reminder of some of the functions we have seen. Hint: they
should be useful for the exercises!

-   `sum(expression)`: aggregate to
    return a sum for a set of records
-   `count(expression)`: aggregate to
    return the size of a set of records
-   `ST_Area(geometry)` returns the
    area of the polygons
-   `ST_AsText(geometry)` returns WKT `text`
-   `ST_Contains(geometry A, geometry B)` returns the true if geometry A contains geometry B
-   `ST_Distance(geometry A, geometry B)` returns the minimum distance between geometry A and
    geometry B
-   `ST_DWithin(geometry A, geometry B, radius)` returns the true if geometry A is radius distance or less from geometry B
-   `ST_GeomFromText(text)` returns `geometry`
-   `ST_Intersects(geometry A, geometry B)` returns the true if geometry A intersects geometry B
-   `ST_Length(linestring)` returns the length of the linestring
-   `ST_Touches(geometry A, geometry B)` returns the true if the boundary of geometry A touches geometry B
-   `ST_Within(geometry A, geometry B)` returns the true if geometry A is within geometry B


Uncomment and run the following cell to install the required packages.


In [5]:
%pip install duckdb leafmap lonboard
import duckdb
import leafmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.8/667.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 16.9 M

In [6]:
url = "https://storage.googleapis.com/qm2/CASA0025/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

Downloading...
From: https://storage.googleapis.com/qm2/CASA0025/nyc_data.db.zip
To: /content/nyc_data.db.zip
100%|██████████| 8.60M/8.60M [00:00<00:00, 8.66MB/s]


Extracting files...


'/content/nyc_data.db.zip'

In [3]:
con = duckdb.connect("nyc_data.db")

con.install_extension("spatial")
con.load_extension("spatial")

In [4]:
con.sql("SHOW TABLES;")

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ nyc_census_blocks   │
│ nyc_homicides       │
│ nyc_neighborhoods   │
│ nyc_streets         │
│ nyc_subway_stations │
└─────────────────────┘

Download the [nyc_data.zip](https://github.com/opengeos/data/raw/main/duckdb/nyc_data.zip) dataset using leafmap. The zip file contains the following datasets. Create a new DuckDB database and import the datasets into the database. Each dataset should be imported into a separate table.

- nyc_census_blocks
- nyc_homicides
- nyc_neighborhoods
- nyc_streets
- nyc_subway_stations

1. **What subway station is in \'Little Italy\'? What subway route is it on?**

In [7]:
con.sql("""

SELECT s.NAME, s.ROUTES
FROM nyc_subway_stations s
JOIN (SELECT geom
	FROM nyc_neighborhoods
	WHERE name = 'Little Italy') n
ON ST_Intersects(s.geom, n.geom)
;
""")

┌───────────┬─────────┐
│   NAME    │ ROUTES  │
│  varchar  │ varchar │
├───────────┼─────────┤
│ Spring St │ 6       │
└───────────┴─────────┘

2. **What are all the neighborhoods served by the 6-train?** (Hint: The `routes` column in the `nyc_subway_stations` table has values like \'B,D,6,V\' and \'C,6\')


In [11]:
con.sql("""
SELECT DISTINCT n.NAME
FROM nyc_neighborhoods n
JOIN (SELECT geom
	FROM nyc_subway_stations
	WHERE ROUTES LIKE '%6%') s
ON ST_Intersects(n.geom, s.geom)
;
""")

┌────────────────────┐
│        NAME        │
│      varchar       │
├────────────────────┤
│ Gramercy           │
│ Murray Hill        │
│ Soundview          │
│ Upper East Side    │
│ Financial District │
│ Hunts Point        │
│ East Harlem        │
│ Little Italy       │
│ Yorkville          │
│ Chinatown          │
│ Mott Haven         │
│ Greenwich Village  │
│ Midtown            │
│ Parkchester        │
│ South Bronx        │
├────────────────────┤
│      15 rows       │
└────────────────────┘

3. **After 9/11, the \'Battery Park\' neighborhood was off limits for several days. How many people had to be evacuated?**

In [30]:
con.sql("""
SELECT SUM(c.POPN_TOTAL)
FROM nyc_census_blocks c
JOIN (SELECT geom
	FROM nyc_neighborhoods
	WHERE NAME = 'Battery Park') n
ON ST_Intersects(n.geom, c.geom)

;
""")

┌───────────────────┐
│ sum(c.POPN_TOTAL) │
│      int128       │
├───────────────────┤
│             17153 │
└───────────────────┘

4. **What neighborhood has the highest population density (persons/km2)?**


In [28]:
con.sql("""
SELECT n.NAME,
SUM(c.POPN_TOTAL)/(ST_area(n.geom)/100000) AS pop_density
FROM nyc_census_blocks c
JOIN nyc_neighborhoods n
ON ST_Intersects(c.geom, n.geom)
GROUP BY  n.name, n.geom
ORDER BY pop_density DESC
;
""")

┌─────────────────────────────┬────────────────────┐
│            NAME             │    pop_density     │
│           varchar           │       double       │
├─────────────────────────────┼────────────────────┤
│ North Sutton Area           │  6843.513283772679 │
│ East Village                │  5040.448341332535 │
│ Chinatown                   │   4882.51805506297 │
│ Carnegie Hill               │  4854.372540441477 │
│ Upper East Side             │  4852.448774898572 │
│ Little Italy                │  4676.934738855346 │
│ Greenwich Village           │  4283.592043121244 │
│ Gramercy                    │  4181.238571997337 │
│ Morris Heights              │ 4068.7922840688666 │
│ Upper West Side             │ 4015.2489608002365 │
│      ·                      │          ·         │
│      ·                      │          ·         │
│      ·                      │          ·         │
│ Howland Hook                │  92.40400528964409 │
│ Bloomfield-Chelsea-Travis   │  77.4379049981

When you're finished, you can check your answers [here](https://postgis.net/workshops/postgis-intro/joins_exercises.html).

# Ship-to-Ship Transfer Detection

Now for a less structured exercise. We're going to look at ship-to-ship transfers. The idea is that two ships meet up in the middle of the ocean, and one ship transfers cargo to the other. This is a common way to avoid sanctions, and is often used to transfer oil from sanctioned countries to other countries. We're going to look at a few different ways to detect these transfers using AIS data.

In [7]:
%pip install duckdb duckdb-engine jupysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.8/193.8 kB 4.6 MB/s eta 0:00:00


In [8]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///:memory:

In [9]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL spatial;
LOAD spatial;

,Success


## Step 1

Create a spatial database using the following AIS data:

https://storage.googleapis.com/qm2/casa0025_ships.csv

Each row in this dataset is an AIS 'ping' indicating the position of a ship at a particular date/time, alongside vessel-level characteristics.

It contains the following columns:
* `vesselid`: A unique numerical identifier for each ship, like a license plate
* `vessel_name`: The ship's name
* `vsl_descr`: The ship's type
* `dwt`: The ship's Deadweight Tonnage (how many tons it can carry)
* `v_length`: The ship's length in meters
* `draught`: How many meters deep the ship is draughting (how low it sits in the water). Effectively indicates how much cargo the ship is carrying
* `sog`: Speed over Ground (in knots)
* `date`: A timestamp for the AIS signal
* `lat`: The latitude of the AIS signal (EPSG:4326)
* `lon`: The longitude of the AIS signal (EPSG:4326)

Create a table called 'ais' where each row is a different AIS ping, with no superfluous information. Construct a geometry column.

Create a second table called 'vinfo' which contains vessel-level information with no superfluous information.

You can set a spatial index on each of these tables as follows:

`CREATE INDEX index_name ON table_name USING RTREE(geom);`

In [10]:
%%sql

SELECT * FROM 'https://storage.googleapis.com/qm2/casa0025_ships.csv';

,vesselid,vessel_name,vsl_descr,dwt,v_length,draught,sog,date,lat,lon,geom
0,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,5.2,2022-07-25 02:53:29,45.151777,36.513327,POINT (36.5133266666667 45.1517766666667)
1,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:09:37,45.146487,36.520780,POINT (36.52078 45.1464866666667)
2,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:13:58,45.146218,36.521965,POINT (36.521965 45.1462183333333)
3,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.1,2022-07-25 04:16:06,45.145058,36.522020,POINT (36.52202 45.1450583333333)
4,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.0,2022-07-25 05:20:17,45.144933,36.521848,POINT (36.5218483333333 45.1449333333333)
...,...,...,...,...,...,...,...,...,...,...,...
101323,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:16:47,45.091987,36.522157,POINT (36.5221566666667 45.0919866666667)
101324,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:43:48,45.091643,36.522213,POINT (36.5222133333333 45.0916433333333)
101325,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,5.8,2022-08-10 15:04:28,45.100457,36.519397,POINT (36.5193966666667 45.1004566666667)
101326,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,8.3,2022-08-23 06:06:51,45.087527,36.506987,POINT (36.5069866666667 45.0875266666667)


In [11]:
%%sql

INSTALL httpfs;
LOAD httpfs;

,Success


In [12]:
%%sql
CREATE TABLE shiptoship AS SELECT * FROM 'https://storage.googleapis.com/qm2/casa0025_ships.csv';

,Success


In [46]:
%%sql
FROM shiptoship;

,vesselid,vessel_name,vsl_descr,dwt,v_length,draught,sog,date,lat,lon,geom
0,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,5.2,2022-07-25 02:53:29,45.151777,36.513327,POINT (36.5133266666667 45.1517766666667)
1,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:09:37,45.146487,36.520780,POINT (36.52078 45.1464866666667)
2,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:13:58,45.146218,36.521965,POINT (36.521965 45.1462183333333)
3,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.1,2022-07-25 04:16:06,45.145058,36.522020,POINT (36.52202 45.1450583333333)
4,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.0,2022-07-25 05:20:17,45.144933,36.521848,POINT (36.5218483333333 45.1449333333333)
...,...,...,...,...,...,...,...,...,...,...,...
101323,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:16:47,45.091987,36.522157,POINT (36.5221566666667 45.0919866666667)
101324,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:43:48,45.091643,36.522213,POINT (36.5222133333333 45.0916433333333)
101325,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,5.8,2022-08-10 15:04:28,45.100457,36.519397,POINT (36.5193966666667 45.1004566666667)
101326,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,8.3,2022-08-23 06:06:51,45.087527,36.506987,POINT (36.5069866666667 45.0875266666667)


## Step 2

Use a spatial join to identify ship-to-ship transfers in this dataset.
Two ships are considered to be conducting a ship to ship transfer IF:

* They are within 500 meters of each other
* For more than two hours
* And their speed is lower than 1 knot

Some things to consider: make sure you're not joining ships with themselves. Try working with subsets of the data first while you try different things out.

In [14]:
%config SqlMagic.named_parameters="enabled"

In [17]:
%%sql

CREATE OR REPLACE TABLE ais AS
SELECT
  vesselid,
  date::TIMESTAMP AS ts,
  sog,
  draught,
  ST_Point(lon, lat) AS geom
FROM shiptoship
WHERE lon IS NOT NULL AND lat IS NOT NULL;

,Success


In [50]:
%%sql
FROM ais

,vesselid,ts,sog,draught,geom
0,350053,2022-07-25 02:53:29,5.2,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,350053,2022-07-25 03:09:37,0.7,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,350053,2022-07-25 03:13:58,0.7,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,350053,2022-07-25 04:16:06,0.1,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,350053,2022-07-25 05:20:17,0.0,3.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
...,...,...,...,...,...
101323,217531,2022-08-10 14:16:47,0.1,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101324,217531,2022-08-10 14:43:48,0.1,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101325,217531,2022-08-10 15:04:28,5.8,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
101326,217531,2022-08-23 06:06:51,8.3,4.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [18]:
%%sql

CREATE TABLE vinfo AS
SELECT DISTINCT
  vesselid,
  vessel_name,
  vsl_descr,
  dwt,
  v_length,

FROM shiptoship
;

,Success


In [19]:
%%sql
FROM vinfo

,vesselid,vessel_name,vsl_descr,dwt,v_length
0,350053,30 Let Pobedy,general cargo,5150.0,NaN
1,323648,A Line,bulk carrier,12259.0,109.0
2,330665,Adafera,crude oil tanker,105215.0,226.0
3,269668,Adam- A,bulk carrier,28458.0,163.0
4,285151,Adler,roll on roll off with container capacity,7331.0,120.0
...,...,...,...,...,...
830,312965,Zeynep C,bulk carrier,53806.0,183.0
831,272505,Zhadeit,general cargo,3775.0,139.0
832,10528606,Zhibek Zholy,general cargo,7146.0,140.0
833,356770,Zoi XL,bulk carrier,82489.0,229.0


In [30]:
%%sql

WITH close_pairs AS (
  SELECT
    a1.vesselid AS ship1,
    a2.vesselid AS ship2,
    a1.ts AS t
  FROM ais a1
  JOIN ais a2
    ON a1.vesselid < a2.vesselid
   AND ABS(epoch(a1.ts) - epoch(a2.ts)) <= 300   -- ±5 min (misma hora aprox)
   AND a1.sog < 1
   AND a2.sog < 1
   AND ST_Distance(
         ST_Transform(a1.geom,'EPSG:4326','EPSG:3857'),
         ST_Transform(a2.geom,'EPSG:4326','EPSG:3857')
       ) <= 500

)
SELECT *
FROM close_pairs;

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ship1,ship2,t
0,350053,352316,2022-08-14 03:08:36
1,350053,370492,2022-08-05 09:55:22
2,350053,370492,2022-08-05 09:55:22
3,350053,370492,2022-08-05 11:05:53
4,350053,370492,2022-08-05 12:08:02
...,...,...,...
4264,356770,10237066,2022-08-24 07:14:23
4265,356770,10237066,2022-08-24 08:18:34
4266,356770,10237066,2022-08-24 13:50:54
4267,356770,11904057,2022-08-19 06:40:44


In [26]:
%%sql

FROM st_transfers

,v1,v2,ep_id,start_time,end_time,duration_minutes,n_matches
